In [1]:
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc

REPO_ROOT = Path("../..").resolve()
NOMBRE = "A1606b0e6_0_62"
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
OUT_DIR = REPO_ROOT / "outputs" / "tactico"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(v) for k, v in data["color_equipo_bgr"].items()}
col_eq = {eq: tuple(np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}  # RGB

MARGEN = 5.0
def en_campo(c):
    return c is not None and -MARGEN <= c[0] <= 105 + MARGEN and -MARGEN <= c[1] <= 68 + MARGEN

# Posiciones de jugadores de campo por frame y equipo (excluye porteros para métricas de bloque)
def jugadores_frame(f_idx, equipo, incluir_portero=False):
    roles = ("jugador", "portero") if incluir_portero else ("jugador",)
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") in roles and d.get("equipo") == equipo and en_campo(d.get("campo"))]

print(f"JSON: {JSON_IN.name} | Frames: {N_FRAMES}")
print(f"Equipos: A={color_equipo_bgr[0]}, B={color_equipo_bgr[1]}")

JSON: integrado_A1606b0e6_0_62.json | Frames: 749
Equipos: A=(220, 250, 254), B=(119, 182, 254)


In [2]:
# ---- Filtrar fragmentos de tracking: quedarse con jugadores de presencia sólida ----
# El tracking fragmenta algún jugador (un ID se pierde y se crea otro corto). Esos fragmentos
# (pocas detecciones) ensucian las métricas. Se identifican los IDs "titulares" por equipo:
# los que tienen presencia sostenida. Los fragmentos cortos se excluyen del análisis.
from collections import defaultdict

MIN_DET_TITULAR = 200   # un titular aparece en cientos de frames; un fragmento, en pocos

det_por_id_eq = defaultdict(int)
for fr in frames:
    for d in fr:
        if d.get("rol") == "jugador" and d.get("equipo") in (0, 1) and en_campo(d.get("campo")):
            det_por_id_eq[(d["equipo"], d["id"])] += 1

# IDs válidos (titulares) por equipo
ids_validos = {(eq, i) for (eq, i), n in det_por_id_eq.items() if n >= MIN_DET_TITULAR}

print("Jugadores titulares (presencia sólida) por equipo:")
for eq in (0, 1):
    tits = [i for (e, i) in ids_validos if e == eq]
    frags = [i for (e, i), n in det_por_id_eq.items() if e == eq and n < MIN_DET_TITULAR]
    print(f"  Equipo {'AB'[eq]}: {len(tits)} titulares | {len(frags)} fragmentos excluidos {frags if frags else ''}")

# Redefinir jugadores_frame para excluir fragmentos
def jugadores_frame(f_idx, equipo, incluir_portero=False):
    roles = ("jugador", "portero") if incluir_portero else ("jugador",)
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") in roles and d.get("equipo") == equipo
            and en_campo(d.get("campo")) and (equipo, d["id"]) in ids_validos]

Jugadores titulares (presencia sólida) por equipo:
  Equipo A: 10 titulares | 3 fragmentos excluidos [27, 57, 59]
  Equipo B: 10 titulares | 0 fragmentos excluidos 


In [3]:
def dibujar_campo(ax, fondo="#2d5a27"):
    ax.add_patch(Rectangle((0, 0), 105, 68, color=fondo, zorder=0))
    lc, lw = "white", 1.5
    ax.add_patch(Rectangle((0, 0), 105, 68, fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([52.5, 52.5], [0, 68], color=lc, linewidth=lw)
    ax.add_patch(Circle((52.5, 34), 9.15, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((99.5, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.set_xlim(-5, 110); ax.set_ylim(-5, 73)
    ax.set_aspect("equal"); ax.axis("off")
print("dibujar_campo listo")

dibujar_campo listo


In [4]:
# ---- 1. Centroide, dispersión y líneas (defensa/medio/ataque) por equipo ----
# Centroide: posición media del bloque. Dispersión: cómo de juntos juegan (desviación).
# Líneas: agrupando jugadores por su X (avance), se identifican defensa/medio/ataque.

metricas = {0: defaultdict(list), 1: defaultdict(list)}
for f_idx in range(N_FRAMES):
    for eq in (0, 1):
        pts = np.array(jugadores_frame(f_idx, eq))
        if len(pts) < 5:
            continue
        cen = pts.mean(axis=0)
        metricas[eq]["centroide_x"].append(cen[0])
        metricas[eq]["centroide_y"].append(cen[1])
        # dispersión: desviación típica de distancias al centroide
        metricas[eq]["dispersion"].append(np.linalg.norm(pts - cen, axis=1).std())
        # amplitud (ancho en Y) y profundidad (largo en X) del bloque
        metricas[eq]["amplitud"].append(pts[:, 1].max() - pts[:, 1].min())
        metricas[eq]["profundidad"].append(pts[:, 0].max() - pts[:, 0].min())

print("=== MÉTRICAS DE BLOQUE (promedios sobre el vídeo) ===")
for eq in (0, 1):
    nombre = "A" if eq == 0 else "B"
    cx = np.mean(metricas[eq]["centroide_x"])
    cy = np.mean(metricas[eq]["centroide_y"])
    disp = np.mean(metricas[eq]["dispersion"])
    amp = np.mean(metricas[eq]["amplitud"])
    prof = np.mean(metricas[eq]["profundidad"])
    print(f"Equipo {nombre}: centroide=({cx:.1f},{cy:.1f}) | dispersión={disp:.1f}m | "
          f"amplitud={amp:.1f}m | profundidad={prof:.1f}m")

# Visualización: centroide medio + elipse de dispersión de cada equipo
fig, ax = plt.subplots(figsize=(11, 8), dpi=100)
dibujar_campo(ax)
for eq in (0, 1):
    cx, cy = np.mean(metricas[eq]["centroide_x"]), np.mean(metricas[eq]["centroide_y"])
    disp = np.mean(metricas[eq]["dispersion"])
    ax.scatter([cx], [cy], s=300, c=[col_eq[eq]], edgecolors="black", linewidth=2, zorder=6)
    ax.add_patch(Circle((cx, cy), disp, fill=False, edgecolor=col_eq[eq], linewidth=2, linestyle="--"))
    ax.annotate(f"{'AB'[eq]}", (cx, cy), fontsize=14, fontweight="bold", ha="center", va="center")
ax.set_title("Centroide y dispersión media por equipo", color="white", fontsize=13)
fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR / f"centroide_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a")
plt.show()
print(f"\nGuardado: centroide_{NOMBRE}.png")

=== MÉTRICAS DE BLOQUE (promedios sobre el vídeo) ===
Equipo A: centroide=(30.4,33.8) | dispersión=5.1m | amplitud=27.5m | profundidad=24.5m
Equipo B: centroide=(35.2,32.3) | dispersión=6.3m | amplitud=42.2m | profundidad=30.5m

Guardado: centroide_A1606b0e6_0_62.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\231993036.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ---- 2. Mapas de calor de ocupación por equipo ----
from scipy.ndimage import gaussian_filter

def mapa_calor(equipo, bins=(52, 34), sigma=1.5):
    pts = np.array([d["campo"] for fr in frames for d in fr
                    if d.get("rol") == "jugador" and d.get("equipo") == equipo and en_campo(d.get("campo"))])
    H, xe, ye = np.histogram2d(pts[:, 0], pts[:, 1], bins=bins, range=[[0, 105], [0, 68]])
    return gaussian_filter(H.T, sigma=sigma)

fig, axes = plt.subplots(1, 2, figsize=(20, 7), dpi=100)
for eq, ax in zip((0, 1), axes):
    dibujar_campo(ax, fondo="#1a3a17")
    heat = mapa_calor(eq)
    ax.imshow(heat, extent=[0, 105, 0, 68], origin="lower", cmap="hot",
              alpha=0.6, aspect="equal", zorder=1)
    ax.set_title(f"Mapa de calor - Equipo {'AB'[eq]}", color="white", fontsize=13)
fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR / f"calor_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a")
plt.show()
print(f"Guardado: calor_{NOMBRE}.png")

Guardado: calor_A1606b0e6_0_62.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\159599017.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ---- 3. Posesión con NEUTRO RELAJADO (dist_control=10, franja_disputa=1.0) ----
posesion = {0: 0, 1: 0, "neutro": 0}
zona_posesion = {0: [0, 0, 0], 1: [0, 0, 0]}

for f_idx in range(N_FRAMES):
    bpos = next((d["campo"] for d in frames[f_idx]
                 if d.get("rol") == "balon" and en_campo(d.get("campo"))), None)
    if bpos is None:
        continue
    dmin = {0: 999, 1: 999}
    for d in frames[f_idx]:
        if d.get("rol") == "jugador" and d.get("equipo") in (0, 1) and en_campo(d.get("campo")):
            dd = np.hypot(d["campo"][0] - bpos[0], d["campo"][1] - bpos[1])
            if dd < dmin[d["equipo"]]:
                dmin[d["equipo"]] = dd
    d0, d1 = dmin[0], dmin[1]
    if min(d0, d1) > 10 or abs(d0 - d1) < 1.0:
        posesion["neutro"] += 1
    else:
        eq = 0 if d0 < d1 else 1
        posesion[eq] += 1
        tercio = 0 if bpos[0] < 35 else (1 if bpos[0] < 70 else 2)
        zona_posesion[eq][tercio] += 1

total = posesion[0] + posesion[1] + posesion["neutro"]
print("=== POSESIÓN (con balón dividido) ===")
if total > 0:
    print(f"Equipo A: {100*posesion[0]/total:.0f}%  |  "
          f"Neutro: {100*posesion['neutro']/total:.0f}%  |  "
          f"Equipo B: {100*posesion[1]/total:.0f}%")
    print(f"(sobre {total} frames con balón)")
    for eq in (0, 1):
        z = zona_posesion[eq]; tz = sum(z) or 1
        print(f"Equipo {'AB'[eq]} - zonas (def/medio/ataque): "
              f"{100*z[0]/tz:.0f}% / {100*z[1]/tz:.0f}% / {100*z[2]/tz:.0f}%")

col_a = tuple(np.array(color_equipo_bgr[0][::-1]) / 255.0)
col_b = tuple(np.array(color_equipo_bgr[1][::-1]) / 255.0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
ax1.pie([posesion[0], posesion["neutro"], posesion[1]],
        labels=["Equipo A", "Neutro", "Equipo B"],
        colors=[col_a, "#a0a0a0", col_b], autopct="%1.0f%%", textprops={"fontsize": 12})
ax1.set_title("Posesión (con balón dividido)", fontsize=13)
x = np.arange(3); w = 0.35
ax2.bar(x - w/2, zona_posesion[0], w, label="Equipo A", color=col_a, edgecolor="black")
ax2.bar(x + w/2, zona_posesion[1], w, label="Equipo B", color=col_b, edgecolor="black")
ax2.set_xticks(x); ax2.set_xticklabels(["Defensa", "Medio", "Ataque"])
ax2.set_title("Posesión por zona (excluye neutro)"); ax2.legend()
plt.savefig(OUT_DIR / f"posesion_{NOMBRE}.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"Guardado: posesion_{NOMBRE}.png")

=== POSESIÓN (con balón dividido) ===
Equipo A: 5%  |  Neutro: 11%  |  Equipo B: 84%
(sobre 534 frames con balón)
Equipo A - zonas (def/medio/ataque): 17% / 83% / 0%
Equipo B - zonas (def/medio/ataque): 7% / 93% / 0%
Guardado: posesion_A1606b0e6_0_62.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\2236866787.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ---- Formación: ambos equipos ENFRENTADos, porteros en sus porterías ----
from collections import defaultdict
from sklearn.cluster import KMeans

CATALOGO = {
    "4-4-2":[4,4,2], "4-3-3":[4,3,3], "4-2-3-1":[4,2,3,1], "4-5-1":[4,5,1],
    "4-1-4-1":[4,1,4,1], "4-4-1-1":[4,4,1,1], "3-5-2":[3,5,2], "3-4-3":[3,4,3],
    "5-3-2":[5,3,2], "5-4-1":[5,4,1],
}
ATAQUE = {0: "derecha", 1: "izquierda"}

def formacion_equipo(eq):
    det = defaultdict(list)
    for fr in frames:
        for d in fr:
            if d.get("rol") == "jugador" and d.get("equipo") == eq and en_campo(d.get("campo")):
                det[d["id"]].append(d["campo"])
    jug = {i: np.median(ps, axis=0) for i, ps in det.items() if len(ps) >= 200}
    if len(jug) < 8:
        return None, None
    ids = list(jug); P = np.array([jug[i] for i in ids])
    avance = P[:, 0] if ATAQUE[eq] == "derecha" else (105 - P[:, 0])
    orden = np.argsort(avance); n = len(jug)
    mejor, mc = None, 1e9
    for nom, pat in CATALOGO.items():
        if sum(pat) != n: continue
        coste, idx, cen = 0, 0, []
        for c in pat:
            g = orden[idx:idx+c]; cen.append(avance[g].mean()); coste += avance[g].std()*c; idx += c
        if any(cen[i] >= cen[i+1] for i in range(len(cen)-1)): coste += 50
        if coste < mc: mc, mejor = coste, (nom, pat)
    return jug, mejor

def dibujar_campo_completo(ax):
    ax.add_patch(Rectangle((0,0),105,68, color="#2d5a27", zorder=0))
    bl, lw = "white", 1.5
    ax.add_patch(Rectangle((0,0),105,68, fill=False, edgecolor=bl, linewidth=lw))
    ax.plot([52.5,52.5],[0,68], color=bl, linewidth=lw)
    ax.add_patch(Circle((52.5,34),9.15, fill=False, edgecolor=bl, linewidth=lw))
    for spec in [((0,13.84),16.5,40.32),((88.5,13.84),16.5,40.32),((0,24.84),5.5,18.32),((99.5,24.84),5.5,18.32)]:
        ax.add_patch(Rectangle(spec[0],spec[1],spec[2], fill=False, edgecolor=bl, linewidth=lw))
    ax.set_xlim(-5,110); ax.set_ylim(-5,73); ax.set_aspect("equal"); ax.axis("off")

def posiciones_ideales(formacion, ataca):
    """Posiciones canónicas (incluido portero) de una formación, según hacia dónde ataca."""
    patron = CATALOGO[formacion]
    n_lin = len(patron)
    if ataca == "derecha":
        gk_x = 5
        xs_lineas = np.linspace(20, 50, n_lin)   # defensa cerca portería izq, ataque hacia centro
    else:
        gk_x = 100
        xs_lineas = np.linspace(85, 55, n_lin)   # defensa cerca portería der, ataque hacia centro
    posiciones = [("GK", gk_x, 34)]
    for lx, cuantos in zip(xs_lineas, patron):
        ys = np.linspace(10, 58, cuantos + 2)[1:-1] if cuantos > 1 else [34]
        for y in ys:
            posiciones.append(("J", lx, y))
    return posiciones

# --- Figura: arriba posiciones reales (enfrentadas), abajo formación ideal (enfrentada) ---
fig, (ax_real, ax_ideal) = plt.subplots(2, 1, figsize=(13, 18), dpi=100)

# ARRIBA: posiciones reales de ambos equipos en el MISMO campo
dibujar_campo_completo(ax_real)
formaciones = {}
for eq in (0, 1):
    jug, resultado = formacion_equipo(eq)
    formaciones[eq] = resultado
    col = tuple(np.array(color_equipo_bgr[eq][::-1]) / 255.0)
    if jug:
        for i, pos in jug.items():
            ax_real.scatter(pos[0], pos[1], s=220, c=[col], edgecolors="black", linewidth=1.5, zorder=5)
    # portero: real si se detecta, hipotético si no
    gk = next((d["campo"] for fr in frames for d in fr
               if d.get("rol")=="portero" and d.get("equipo")==eq and en_campo(d.get("campo"))), None)
    if gk:
        ax_real.scatter(gk[0], gk[1], s=240, c=[col], edgecolors="white", linewidth=2.5, zorder=6)
    else:
        gk_x = 5 if ATAQUE[eq]=="derecha" else 100
        ax_real.scatter(gk_x, 34, s=240, c=[col], edgecolors="yellow", linewidth=2.5, zorder=6, marker="s")
        ax_real.annotate("GK (hipotético)", (gk_x, 34), fontsize=9, color="yellow", ha="center", va="bottom")
texto = " vs ".join(f"{'AB'[eq]} 1-{formaciones[eq][0]}" for eq in (0,1) if formaciones[eq])
ax_real.set_title(f"Posiciones reales — {texto}", color="white", fontsize=14)

# ABAJO: formación ideal de ambos equipos, ENFRENTADA en el mismo campo
dibujar_campo_completo(ax_ideal)
for eq in (0, 1):
    if not formaciones[eq]:
        continue
    nombre_form = formaciones[eq][0]
    col = tuple(np.array(color_equipo_bgr[eq][::-1]) / 255.0)
    for tipo, x, y in posiciones_ideales(nombre_form, ATAQUE[eq]):
        edge = "white" if tipo == "J" else "yellow"
        ax_ideal.scatter(x, y, s=240, c=[col], edgecolors=edge, linewidth=2, zorder=6)
ax_ideal.set_title(f"Formación ideal enfrentada — {texto}", color="white", fontsize=14)

fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR / f"formacion_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a")
plt.show()

for eq in (0, 1):
    if formaciones[eq]:
        print(f"Equipo {'AB'[eq]}: 1-{formaciones[eq][0]}, ataca {ATAQUE[eq]}")
print(f"Guardado: formacion_{NOMBRE}.png")

Equipo A: 1-5-3-2, ataca derecha
Equipo B: 1-3-4-3, ataca izquierda
Guardado: formacion_A1606b0e6_0_62.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\3683976532.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
